<a href="https://colab.research.google.com/github/irfanali11/sensor-fusion-robustness-study/blob/main/sensor_fusion_robustness_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install torch torchvision opencv-python scipy numpy matplotlib scikit-learn -q

In [11]:
!wget "http://www.utdallas.edu/~kehtar/UTD-MAD/RGB.zip" -O rgb.zip
!wget "http://www.utdallas.edu/~kehtar/UTD-MAD/Inertial.zip" -O inertial.zip

!mkdir -p data/rgb data/inertial
!unzip -q rgb.zip -d data/rgb
!unzip -q inertial.zip -d data/inertial

!ls data/rgb | head -5
!ls data/inertial | head -5

--2026-09-04 22:05:14--  http://www.utdallas.edu/~kehtar/UTD-MAD/RGB.zip
Resolving www.utdallas.edu (www.utdallas.edu)... 3.21.250.42, 3.133.32.155
Connecting to www.utdallas.edu (www.utdallas.edu)|3.21.250.42|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.utdallas.edu/~kehtar/UTD-MAD/RGB.zip [following]
--2026-09-04 22:05:15--  https://www.utdallas.edu/~kehtar/UTD-MAD/RGB.zip
Connecting to www.utdallas.edu (www.utdallas.edu)|3.21.250.42|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://personal.utdallas.edu/~kehtar/UTD-MAD/RGB.zip [following]
--2026-09-04 22:05:16--  https://personal.utdallas.edu/~kehtar/UTD-MAD/RGB.zip
Resolving personal.utdallas.edu (personal.utdallas.edu)... 129.110.46.112
Connecting to personal.utdallas.edu (personal.utdallas.edu)|129.110.46.112|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1106885499 (1.0G) [application/zip]
Saving to: ‘rgb.zip’

In [12]:
!ls data/rgb/RGB | head -5
!ls data/inertial/Inertial | head -5

ls: cannot access 'data/rgb/RGB': No such file or directory
a10_s1_t1_inertial.mat
a10_s1_t2_inertial.mat
a10_s1_t3_inertial.mat
a10_s1_t4_inertial.mat
a10_s2_t1_inertial.mat


In [13]:
!ls data/rgb/RGB | wc -l
!ls data/inertial/Inertial | wc -l

ls: cannot access 'data/rgb/RGB': No such file or directory
0
861


In [14]:
import os
import cv2
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [17]:
def load_video_frames(video_path, num_frames=16, resize=(112, 112)):
    """Load a fixed number of evenly-spaced frames from a video, resized."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return np.zeros((num_frames, *resize, 3), dtype=np.float32)

    idxs = np.linspace(0, max(total - 1, 0), num_frames).astype(int)
    idx_set = set(idxs.tolist())
    current = 0
    grabbed = {}
    while cap.isOpened() and len(grabbed) < len(idx_set):
        ret, frame = cap.read()
        if not ret:
            break
        if current in idx_set:
            frame = cv2.resize(frame, resize)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            grabbed[current] = frame
        current += 1
    cap.release()

    frames = [grabbed.get(i, np.zeros((*resize, 3), dtype=np.float32)) for i in idxs]
    return np.stack(frames, axis=0)  # (num_frames, H, W, 3)


def load_inertial(mat_path, seq_len=100):
    """Load IMU data from .mat file, pad/truncate to fixed sequence length."""
    mat = sio.loadmat(mat_path)
    data = mat["d_iner"].astype(np.float32)  # shape (T, 6): accel(3)+gyro(3)
    if data.shape[0] >= seq_len:
        data = data[:seq_len]
    else:
        pad = np.zeros((seq_len - data.shape[0], data.shape[1]), dtype=np.float32)
        data = np.concatenate([data, pad], axis=0)
    return data  # (seq_len, 6)


def parse_action_label(filename):
    """'a1_s2_t3_color.avi' -> action label 0 (zero-indexed)."""
    base = os.path.basename(filename)
    action_str = base.split("_")[0]  # 'a1'
    return int(action_str.replace("a", "")) - 1

In [19]:
class UTDMHADDataset(Dataset):
    def __init__(self, rgb_dir, inertial_dir, num_frames=16, seq_len=100,
                 corruption_fn=None):
        self.rgb_dir = rgb_dir
        self.inertial_dir = inertial_dir
        self.num_frames = num_frames
        self.seq_len = seq_len
        self.corruption_fn = corruption_fn

        self.samples = []
        for fname in sorted(os.listdir(rgb_dir)):
            if not fname.endswith("_color.avi"):
                continue
            base_id = fname.replace("_color.avi", "")
            inertial_fname = base_id + "_inertial.mat"
            inertial_path = os.path.join(inertial_dir, inertial_fname)
            if os.path.exists(inertial_path):
                self.samples.append((
                    os.path.join(rgb_dir, fname),
                    inertial_path,
                    parse_action_label(fname)
                ))

        print(f"Found {len(self.samples)} paired video+IMU samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, imu_path, label = self.samples[idx]
        video = load_video_frames(video_path, self.num_frames)
        imu = load_inertial(imu_path, self.seq_len)

        if self.corruption_fn is not None:
            video, imu = self.corruption_fn(video, imu)

        video = torch.from_numpy(video).permute(0, 3, 1, 2).float()  # (T,3,H,W)
        imu = torch.from_numpy(imu).float()                           # (seq_len,6)
        return video, imu, label


def get_dataloaders(rgb_dir, inertial_dir, batch_size=8, corruption_fn=None):
    full_dataset = UTDMHADDataset(rgb_dir, inertial_dir, corruption_fn=corruption_fn)
    n = len(full_dataset)
    idxs = list(range(n))
    train_idx, test_idx = train_test_split(idxs, test_size=0.2, random_state=42)

    train_set = torch.utils.data.Subset(full_dataset, train_idx)
    test_set = torch.utils.data.Subset(full_dataset, test_idx)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

In [20]:
class VisionBranch(nn.Module):
    def __init__(self, out_dim=64):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(64, out_dim)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        feat = self.cnn(x).view(B * T, -1)
        feat = self.fc(feat)
        feat = feat.view(B, T, -1).mean(dim=1)
        return feat

In [21]:
class IMUBranch(nn.Module):
    def __init__(self, in_channels=6, out_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, padding=2), nn.ReLU(),
            nn.Conv1d(32, 32, kernel_size=5, padding=2), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc = nn.Linear(32, out_dim)

    def forward(self, x):
        x = x.permute(0, 2, 1)  # (B,6,seq_len)
        feat = self.net(x).squeeze(-1)
        return self.fc(feat)